# Assignment 11: Fine-Tuning GPT-2 on a Custom Text Style
**Goal:** see how fine-tuning makes a pretrained GPT-2 imitate a specific writing style.

**Steps:** 1) load + clean data → 2) generate with base GPT-2 → 3) fine-tune with Hugging Face `Trainer` → 4) generate with the fine-tuned model (same prompt) → 5) compare.

> Run on Colab with a GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!pip install -q transformers datasets accelerate

## 0. Settings (edit these)

In [ ]:
DATA_FILE = "dataset.txt"      # your raw 200-300 line text file
PROMPT    = "Replace this with the SAME prompt you used in Assignment 2"

MODEL_NAME  = "gpt2"           # small GPT-2
OUTPUT_DIR  = "gpt2-finetuned"
EPOCHS      = 4
BATCH_SIZE  = 4
LR          = 5e-5
MAX_LEN     = 128
SEED        = 42

# Generation settings: identical for base and fine-tuned so the comparison is fair
GEN_KWARGS = dict(max_new_tokens=80, do_sample=True, temperature=0.8,
                  top_k=50, top_p=0.95, repetition_penalty=1.0)
NUM_SAMPLES = 3

## 1. Load the dataset

In [ ]:
# In Colab, upload your file first (folder icon on the left, or run the lines below)
try:
    from google.colab import files
    import os
    if not os.path.exists(DATA_FILE):
        uploaded = files.upload()
        DATA_FILE = list(uploaded.keys())[0]
except ImportError:
    pass

with open(DATA_FILE, encoding="utf-8") as f:
    raw_lines = f.read().splitlines()
print("Raw lines:", len(raw_lines))
print("\n".join(raw_lines[:5]))

## 2. Clean the dataset
Removes timestamps, usernames, URLs, HTML/markdown formatting and non-language characters, then drops empty/duplicate lines.
If your dialogue lines start with speaker names you want to **keep** (e.g. `Alice: hi`), set `STRIP_SPEAKER_LABELS = False`.

In [ ]:
import re

STRIP_SPEAKER_LABELS = True   # removes "Name:" or "<name>" at the start of a line
KEEP_ASCII_ONLY      = True   # drops emojis / symbols from other scripts
MIN_WORDS            = 3

def clean_line(line):
    s = line
    s = re.sub(r"https?://\S+|www\.\S+", "", s)                              # URLs
    s = re.sub(r"<[^>]+>", " ", s) if not STRIP_SPEAKER_LABELS else re.sub(r"<[^>]+>", "", s)  # HTML / <user> tags
    s = re.sub(r"\[?\(?\d{1,2}[:.]\d{2}([:.]\d{2})?\s?(AM|PM|am|pm)?\)?\]?", "", s)  # timestamps
    s = re.sub(r"\d{4}-\d{2}-\d{2}|\d{1,2}/\d{1,2}/\d{2,4}", "", s)         # dates
    s = re.sub(r"@\w+", "", s)                                                  # @usernames
    if STRIP_SPEAKER_LABELS:
        s = re.sub(r"^\s*[\w .'-]{1,25}:\s+", "", s)                           # "Name: "
    s = re.sub(r"[*_`#>~|]+", "", s)                                             # markdown chars
    s = re.sub(r"\[(verse|chorus|bridge|intro|outro)[^\]]*\]", "", s, flags=re.I)  # lyric section tags
    if KEEP_ASCII_ONLY:
        s = s.encode("ascii", "ignore").decode()
    s = re.sub(r"\s+", " ", s).strip()
    return s

cleaned, seen = [], set()
for l in raw_lines:
    c = clean_line(l)
    if len(c.split()) >= MIN_WORDS and c.lower() not in seen:
        seen.add(c.lower()); cleaned.append(c)

print(f"Kept {len(cleaned)} of {len(raw_lines)} lines")
print("\n".join(cleaned[:5]))
with open("dataset_clean.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(cleaned))

## 3. Generate with the ORIGINAL GPT-2 (before fine-tuning)

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer,
                          TrainingArguments, DataCollatorForLanguageModeling, set_seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def generate(model, prompt):
    set_seed(SEED)   # same seed → fair comparison
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, num_return_sequences=NUM_SAMPLES,
                             pad_token_id=tokenizer.eos_token_id, **GEN_KWARGS)
    return [tokenizer.decode(o, skip_special_tokens=True) for o in out]

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
base_outputs = generate(base_model, PROMPT)

for i, o in enumerate(base_outputs, 1):
    print(f"--- BASE sample {i} ---\n{o}\n")

with open("base_model_output.txt", "w", encoding="utf-8") as f:
    f.write(f"PROMPT: {PROMPT}\n\n" + "\n\n".join(f"--- Sample {i} ---\n{o}" for i, o in enumerate(base_outputs, 1)))
del base_model; torch.cuda.empty_cache()

## 4. Fine-tune with the Hugging Face `Trainer`

In [ ]:
from datasets import Dataset

ds = Dataset.from_dict({"text": cleaned}).train_test_split(test_size=0.1, seed=SEED)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = ds.map(tokenize, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=10,
    logging_steps=10,
    save_strategy="no",
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(model=model, args=args,
                  train_dataset=tokenized["train"],
                  eval_dataset=tokenized["test"],
                  data_collator=collator)

trainer.train()
print("Eval:", trainer.evaluate())

trainer.save_model(OUTPUT_DIR)          # saves the checkpoint (submit this)
tokenizer.save_pretrained(OUTPUT_DIR)

## 5. Generate with the FINE-TUNED model (same prompt, same settings)

In [ ]:
ft_model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR).to(device)
ft_outputs = generate(ft_model, PROMPT)

for i, o in enumerate(ft_outputs, 1):
    print(f"--- FINE-TUNED sample {i} ---\n{o}\n")

with open("finetuned_model_output.txt", "w", encoding="utf-8") as f:
    f.write(f"PROMPT: {PROMPT}\n\n" + "\n\n".join(f"--- Sample {i} ---\n{o}" for i, o in enumerate(ft_outputs, 1)))

## 6. Before-and-after comparison

In [ ]:
import re, statistics, pandas as pd

def stats(texts):
    words = [w.lower() for t in texts for w in re.findall(r"[A-Za-z']+", t)]
    sents = [s for t in texts for s in re.split(r"[.!?\n]+", t) if s.strip()]
    trigrams = list(zip(words, words[1:], words[2:]))
    return {
        "avg words / sample":      round(len(words) / len(texts), 1),
        "avg sentence/line length": round(statistics.mean(len(s.split()) for s in sents), 1),
        "vocab diversity (TTR)":   round(len(set(words)) / max(len(words), 1), 3),
        "repeated trigram rate":   round(1 - len(set(trigrams)) / max(len(trigrams), 1), 3),
        "avg line breaks / sample": round(sum(t.count("\n") for t in texts) / len(texts), 1),
    }

table = pd.DataFrame({"Base GPT-2": stats(base_outputs), "Fine-tuned": stats(ft_outputs)})
display(table)

# Dataset reference stats, to see which model is closer to your style
print("Training data for reference:", stats(["\n".join(cleaned)]))

print("\n=== SIDE BY SIDE (sample 1) ===")
print("BASE:\n", base_outputs[0], "\n\nFINE-TUNED:\n", ft_outputs[0])

with open("before_after_comparison.md", "w", encoding="utf-8") as f:
    f.write(f"# Before vs After\n\n**Prompt:** {PROMPT}\n\n## Metrics\n\n{table.to_markdown()}\n\n")
    f.write("## Base model (sample 1)\n\n" + base_outputs[0] + "\n\n## Fine-tuned model (sample 1)\n\n" + ft_outputs[0] + "\n\n")
    f.write("## Analysis (write your own)\n\n1. Difference 1: ...\n2. Difference 2: ...\n")

## 7. Your analysis (write 2+ differences in your own words)
Look at the outputs and metrics above, and describe differences such as:
- **Vocabulary / word choice:** does the fine-tuned model reuse words and phrases from your dataset?
- **Tone / voice:** formal and generic (base) vs. matching your source's mood or personality?
- **Structure / formatting:** line lengths, line breaks, rhyme, dialogue turns, punctuation habits.
- **Coherence / repetition:** did it become more repetitive or more on-topic?
- **Overfitting:** if it copies lines from the dataset verbatim, mention it (a small dataset makes this likely).

**Difference 1:** *…*

**Difference 2:** *…*

## 8. Download everything for submission

In [ ]:
import shutil
shutil.make_archive("gpt2-finetuned-checkpoint", "zip", OUTPUT_DIR)
try:
    from google.colab import files
    for f in ["dataset.txt", "dataset_clean.txt", "base_model_output.txt",
              "finetuned_model_output.txt", "before_after_comparison.md",
              "gpt2-finetuned-checkpoint.zip"]:
        try: files.download(f)
        except Exception as e: print("Skip", f, e)
except ImportError:
    print("Files saved in the working directory.")